IMPORT LIBRARIES

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Load all four datasets

In [3]:
students = pd.read_csv('C:\\Users\\vikes\\Favorites\\Links\\100daysinC\\explo_project\\Students_Raw.csv')
courses = pd.read_csv('C:\\Users\\vikes\\Favorites\\Links\\100daysinC\\explo_project\\Course_Catalogue.csv')
interactions = pd.read_csv('C:\\Users\\vikes\\Favorites\\Links\\100daysinC\\explo_project\\Student_Course_Interactions.csv')
dictionary = pd.read_csv('C:\\Users\\vikes\\Favorites\\Links\\100daysinC\\explo_project\\Data_Dictionary.csv')


Dimensions of each dataset

In [4]:
print("Students Dataset:", students.shape)
print("Courses Dataset:", courses.shape)
print("Interactions Dataset:", interactions.shape)
print("Dictionary Dataset:", dictionary.shape)

Students Dataset: (1020, 33)
Courses Dataset: (178, 20)
Interactions Dataset: (5869, 10)
Dictionary Dataset: (63, 9)


Inspecting the dataset

In [5]:
students.head()
courses.head()
interactions.head()
dictionary.head()

,Sheet,Column,Description,Expected_Data_Type,Valid_Range_Or_Allowed_Values,Nullable,Role,Notes,Injected_Issue_Types_In_Raw_Data
0,Students_Raw,Student_ID,Unique student identifier (canonical form STU ...,String,STU0001-STU0950 (regex ^STU\d{4}$),No,ID,Duplicated / malformed IDs injected; not a mod...,Inconsistent label / spelling; Invalid / out-o...
1,Students_Raw,Batch_Year,Admission-batch / academic-year cohort aligned...,Integer,"2023, 2024, 2025 (Sem 3-4 -> 2025, Sem 5 -> 20...",Yes,Feature,NaN,Contradictory entry; Incorrect data type / for...
2,Students_Raw,Semester,Current semester of the student,Integer,3-7 (catalogue offers courses for Sem 3-8; val...,Yes,Feature,Determines which catalogue courses are eligible,Incorrect data type / format; Invalid / out-of...
3,Students_Raw,Branch,Programme / specialisation,Categorical,"32 canonical values, e.g. ""B.Tech CSE - Data S...",Yes,Feature,High-cardinality; needs label standardisation,Extra whitespace; Inconsistent label / spellin...
4,Students_Raw,School,UPES school the branch belongs to,Categorical,"SOCS, SOAE, SOB, SOLSH, SOL, SOD, SOHST",Yes,Feature,Derived from Branch; mismatches injected,Contradictory entry; Extra whitespace; Inconsi...


column names and datatypes

In [6]:
print("Students Dataset:")
students.info()
print("\nCourses Dataset:")
courses.info()
print("\nInteractions Dataset:")
interactions.info()
print("\nDictionary Dataset:")
dictionary.info()

Students Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 33 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   Student_ID                     1020 non-null   object
 1   Batch_Year                     1005 non-null   object
 2   Semester                       1020 non-null   object
 3   Branch                         1004 non-null   object
 4   School                         999 non-null    object
 5   Campus                         1008 non-null   object
 6   Age                            990 non-null    object
 7   Gender                         1002 non-null   object
 8   Class12_Stream                 980 non-null    object
 9   CGPA                           959 non-null    object
 10  Previous_Semester_SGPA         966 non-null    object
 11  Attendance_Percentage          970 non-null    object
 12  Weekly_Study_Hours             962 non-null 

Check missing values

In [7]:
datasets = {
    'Students': students,
    'Courses': courses,
    'Interactions': interactions,
    'Dictionary': dictionary
}

for name, dataset in datasets.items():
    print(f"\n{name} Dataset:")
    print("Total missing values:", dataset.isnull().sum().sum())
    print("Missing values per column:")
    print(dataset.isnull().sum()[dataset.isnull().sum() > 0])


Students Dataset:
Total missing values: 1455
Missing values per column:
Batch_Year                       15
Branch                           16
School                           21
Campus                           12
Age                              30
Gender                           18
Class12_Stream                   40
CGPA                             61
Previous_Semester_SGPA           54
Attendance_Percentage            50
Weekly_Study_Hours               58
Num_Certifications               43
Internship_Experience            76
Interests                        42
Secondary_Interests             190
Hobbies                          63
Technical_Skills                 50
Skill_Level                      40
Career_Goal                      66
Learning_Preference              31
Preferred_Difficulty             45
Workload_Preference              50
Preferred_Slot                   59
Confusion_Level                  57
Confusion_Reason                 59
Selection_Confidence       

Check duplicate records

In [8]:
for name, dataset in datasets.items():
    print(f"\n{name} Dataset:")
    print("Exact Duplicate rows:", dataset.duplicated().sum())


Students Dataset:
Exact Duplicate rows: 40

Courses Dataset:
Exact Duplicate rows: 0

Interactions Dataset:
Exact Duplicate rows: 114

Dictionary Dataset:
Exact Duplicate rows: 0


Check Duplicate ids

In [9]:
print("Duplicate student ids:", students['Student_ID'].duplicated().sum())
print("Duplicate course ids:", courses['Course_ID'].duplicated().sum())
print("Duplicate interaction ids:", interactions['Interaction_ID'].duplicated().sum())

Duplicate student ids: 60
Duplicate course ids: 0
Duplicate interaction ids: 136


Check categorical inconsistencies

In [10]:
print("Gender values:")
print(students['Gender'].value_counts(dropna=False))

print("\nCampus values:")
print(students['Campus'].value_counts(dropna=False))

print("\nSkill Level values:")
print(students['Skill_Level'].value_counts(dropna=False))

print("\nLearning Preferences values:")
print(students['Learning_Preference'].value_counts(dropna=False))

Gender values:
Gender
Male              425
Female            238
Male               66
male               46
MALE               41
M                  38
female             28
Femal              22
F                  19
FEMALE             19
NaN                18
Female             11
 Male              10
Not Available       4
 Female             3
male                3
  Male              3
-                   2
Other               2
 Male               2
Others              2
Other               1
O                   1
M                   1
  MALE              1
 NA                 1
 Female             1
  Female            1
 MALE               1
 M                  1
Female\t            1
 Not Available      1
  FEMALE            1
Male\t              1
 F                  1
  M                 1
Unknown             1
?                   1
F                   1
Name: count, dtype: int64

Campus values:
Campus
Bidholi                   439
Kandoli                   230
BIDHOLI    

Check Numerical columns

In [11]:
students.describe()

,Num_Prior_Exploratory_Courses
count,1020.000000
mean,1.291176
std,2.020144
min,-3.000000
25%,0.000000
50%,1.000000
75%,2.000000
max,40.000000


Detect hidden missing values

In [12]:
#check common missing values text entries
missing_tokens= ['NA', 'N/A', 'na', 'n/a', 'null', '-','?','']
for name, df in datasets.items():
    print(f"\n---{name}---")
    for col in df.select_dtypes(include="object").columns:
        missing_count = df[col].isin(missing_tokens).sum()
        if missing_count > 0:
            print(f"Column '{col}' has {missing_count} entries that match common missing value tokens.")


---Students---
Column 'Batch_Year' has 5 entries that match common missing value tokens.
Column 'Branch' has 2 entries that match common missing value tokens.
Column 'School' has 6 entries that match common missing value tokens.
Column 'Campus' has 2 entries that match common missing value tokens.
Column 'Age' has 2 entries that match common missing value tokens.
Column 'Gender' has 3 entries that match common missing value tokens.
Column 'Class12_Stream' has 5 entries that match common missing value tokens.
Column 'CGPA' has 8 entries that match common missing value tokens.
Column 'Previous_Semester_SGPA' has 7 entries that match common missing value tokens.
Column 'Attendance_Percentage' has 6 entries that match common missing value tokens.
Column 'Weekly_Study_Hours' has 6 entries that match common missing value tokens.
Column 'Num_Certifications' has 4 entries that match common missing value tokens.
Column 'Internship_Experience' has 1 entries that match common missing value token

Check numerical values againat valid ranges

In [14]:
#Check values outside expected ranges

range_checks= {
    "Age": (17,30),
    "CGPA": (0,10),
    "Previous_Semester_SGPA": (0,10),
    "Attendance_Percentage": (0,100),
    "Weekly_Study_Hours": (0,60),
    "Num_Prior_Exploratory_Courses": (0,6),
    "Num_Certifications": (0,8),
    "Confusion_Level": (1,10),
    "Selection_Confidence": (1,10),
    "Previous_Course_Satisfaction": (1,5)
}

for col, (low,high) in range_checks.items():
    #convert values to numeric temporarily
    numeric_col= pd.to_numeric(students[col], errors='coerce')

    #count invalid values
    invalid= students[(numeric_col < low) | (numeric_col > high)]

    #count values that could not be converted to numeric
    non_numeric= students[col].notna() & numeric_col.isna()
    print(f"\nColumn '{col}':")
    print("Out of range values:", len(invalid))
    print("Non-numeric values:", non_numeric.sum())
    print("-----------------------")


Column 'Age':
Out of range values: 16
Non-numeric values: 27
-----------------------

Column 'CGPA':
Out of range values: 13
Non-numeric values: 56
-----------------------

Column 'Previous_Semester_SGPA':
Out of range values: 4
Non-numeric values: 12
-----------------------

Column 'Attendance_Percentage':
Out of range values: 13
Non-numeric values: 66
-----------------------

Column 'Weekly_Study_Hours':
Out of range values: 14
Non-numeric values: 35
-----------------------

Column 'Num_Prior_Exploratory_Courses':
Out of range values: 11
Non-numeric values: 0
-----------------------

Column 'Num_Certifications':
Out of range values: 13
Non-numeric values: 10
-----------------------

Column 'Confusion_Level':
Out of range values: 18
Non-numeric values: 28
-----------------------

Column 'Selection_Confidence':
Out of range values: 17
Non-numeric values: 26
-----------------------

Column 'Previous_Course_Satisfaction':
Out of range values: 13
Non-numeric values: 37
------------------

identify inconsistent categorical values

In [15]:
#Check for categorical columns in all datasets
for name, df in datasets.items():
    print(f"\n---{name}---")
    categorical_cols = df.select_dtypes(include="object").columns
    for col in categorical_cols:
        print(f"\n{col}")
        print("Unique values:", df[col].nunique())
        print("Sample values:")
        print(df[col].dropna().unique()[:15])  # Display first 15 unique values


---Students---

Student_ID
Unique values: 960
Sample values:
['STU0001' 'STU0002' 'STU0003' 'STU1350' 'STU0004' 'STU0005' 'STU0006'
 'STU0007' 'STU0008' 'STU0009' 'STU0010' 'STU0011' 'STU0012' 'STU0013'
 'STU0811']

Batch_Year
Unique values: 13
Sample values:
['2023' '2025' '2024' 'AY2023' 'AY2025' '2035' '1999' '?' '2205' '0'
 'Not Available' ' ' '-']

Semester
Unique values: 23
Sample values:
['5' '3' '6' '7' '4' '9' '5.0' 'III' '3.0' 'Sem 3' 'Sem 5' '3rd' '6th'
 '12' 'VII']

Branch
Unique values: 203
Sample values:
['B.Tech CSE - AI & ML' 'B.Tech CSE - Core' 'B.Tech Chemical Engineering'
 'BBA - Foreign Trade' 'BBA - Human Resources' 'B.Tech CSE - Data Science'
 'BBA LLB' 'BTech CSE - Cyber Security' 'B.Tech CSE AI & ML'
 'BA English & Media' 'B.Sc Economics' 'B.Tech Electrical Engineering'
 'BBA - FINANCE' 'B.Tech Electronics & Communication'
 'B.Tech Applied Petroleum Engineering']

School
Unique values: 64
Sample values:
['SOCS' 'S.O.C.S' 'SoAE ' 'SOB' 'SOL' 'SOLSH' 'SoCS' 'SoAE

Check dataset keys

In [16]:
print("Students - Student_ID")
print("Unique:", students['Student_ID'].nunique())
print("Missing:", students['Student_ID'].isnull().sum())

print("\nInteractions - Student_ID")
print("Unique:", interactions['Student_ID'].nunique())
print("Missing:", interactions['Student_ID'].isnull().sum())

print("\nCourses - Course_ID")
print("Unique:", courses['Course_ID'].nunique())
print("Missing:", courses['Course_ID'].isnull().sum())

Students - Student_ID
Unique: 960
Missing: 0

Interactions - Student_ID
Unique: 1106
Missing: 0

Courses - Course_ID
Unique: 178
Missing: 0


inspect the actual problematic values

In [17]:
for col, (low, high) in range_checks.items():
    numeric_col = pd.to_numeric(students[col], errors='coerce')
    invalid_range = students[(numeric_col < low) | (numeric_col > high)] [col]
    non_numeric = students[students[col].notna() & numeric_col.isna()] [col]
    print(f"\nColumn '{col}':")
    print("Out of range values:", invalid_range.unique()[:10])
    print("Non-numeric values:", non_numeric.unique()[:10])
    print("-----------------------")


Column 'Age':
Out of range values: ['31' '35' '62' '40' '12' '99']
Non-numeric values: ['20 yrs' '21 yrs' '18 yrs' 'twenty' 'nineteen' '-' ' ' 'twenty-two'
 '19 yrs' '22 yrs']
-----------------------

Column 'CGPA':
Out of range values: ['12.0' '92.0' '100' '45.0' '85.4' '-1.0' '-8.1' '10.5']
Non-numeric values: ['7,66' '7,31' '7.89 CGPA' '7,51' '6,44' '8,21' '7.2 CGPA' '7.62 CGPA'
 '8,06' '8.47/10']
-----------------------

Column 'Previous_Semester_SGPA':
Out of range values: ['11.4' '95.0']
Non-numeric values: ['-' 'Not Available' 'Unknown' ' ' '?']
-----------------------

Column 'Attendance_Percentage':
Out of range values: ['250' '1000' '-5' '105' '120']
Non-numeric values: ['75%' ' ' '70%' '82 %' '87%' '86%' '75 %' '74 %' '71%' '78%']
-----------------------

Column 'Weekly_Study_Hours':
Out of range values: ['80' '62' '-4' '168' '200' '70' '999' '90']
Non-numeric values: ['Unknown' '24 hrs' 'Not Available' '18 hrs' '16 hrs' '-' '10 hrs'
 '12 hrs' '21 hrs' '6 hrs']
------------

Check relationships between datasets

In [18]:
#Standardize IDs temporarily for comparison
student_ids = set(students['Student_ID'].dropna().astype(str).str.strip())
interaction_student_ids = set(interactions['Student_ID'].dropna().astype(str).str.strip())
course_codes = set(courses['Course_ID'].dropna().astype(str).str.strip())
interaction_course_codes = set(interactions['Course_ID'].dropna().astype(str).str.strip())

#Student ID comparison
print("Student IDs in Interactions but not in Students:", len(interaction_student_ids - student_ids))
print("Student IDs in Students but not in Interactions:", len(student_ids - interaction_student_ids))

#Course code comparison
print("\nCourse Codes in Interactions but not in Catalogue:", len(interaction_course_codes - course_codes))
print("Course Codes in Catalogue but not in Interactions:", len(course_codes - interaction_course_codes))

Student IDs in Interactions but not in Students: 153
Student IDs in Students but not in Interactions: 25

Course Codes in Interactions but not in Catalogue: 0
Course Codes in Catalogue but not in Interactions: 12


Inspect duplicate student IDs

In [20]:
#Find repeated Student IDs
duplicate_students = students[students['Student_ID'].duplicated(keep=False)]
print("Total rows with repeated Student IDs:", len(duplicate_students))
print("\nRepeated Student IDs:")
print(duplicate_students['Student_ID'].value_counts().head(15))  # Display top 15 repeated Student IDs

#inspect the complete records for those IDs
duplicate_students.head(10)

Total rows with repeated Student IDs: 119

Repeated Student IDs:
Student_ID
STU0422    3
STU0811    2
stu0642    2
STU0524    2
STU0340    2
STU0342    2
STU0380    2
STU0385    2
STU0387    2
STU0600    2
STU0439    2
STU0611    2
STU0829    2
STU0921    2
STU0453    2
Name: count, dtype: int64


,Student_ID,Batch_Year,Semester,Branch,School,Campus,Age,Gender,Class12_Stream,CGPA,...,Preferred_Difficulty,Workload_Preference,Preferred_Slot,Confusion_Level,Confusion_Reason,Selection_Confidence,Previous_Course_Satisfaction,Survey_Timestamp,Selected_Course_Code,Selected_Exploratory_Course
14,STU0811,2023,6,BBA - FINANCE,SOB,kandoli,21,M,Arts/Humanities,"7,66",...,Easy,Medium,No Preference,3,Not Available,8/10,3,2026-09-19 01:52:48,SCS20A001,Data Visualization
16,STU0015,2024,5,B.Tech Applied Petroleum Engineering,SOAE,Bidholi,20,MALE,PCM,"7,31",...,Moderate,HIGH,No Preference,7,Too many options,4,3,23-09-26,LSHP3038,"History of Modern India, 1857-1947"
29,STU0332,2025,3,b.sc economics,SoLSH,Kandoli,21,Male,Humanities,6.4,...,Easy,High,No preference,5,Too many options,9,3,2026-10-14 06:01:33,CLCC2044,"The Bharatiya Nyaya Sanhita, 2023"
30,STU0028,2024,5,NaN,SOCS,bidholi,20 yrs,Male,PCM,6.96,...,Easy,Low,Slot 2,7,Unclear course content,3,3,46319.05231481481,DIGB2003P,Introduction to Cloud Computing for Managers
32,STU0030,2025,3,B.Tech CSE - Data Science,SOAE,Kandoli,20,Male,PCMB,8.02,...,Challenging,Low,No Preference,5,Unclear course content,NaN,3,2026-09-22 10:02:06,PEGS2057,Remote Sensing and GIS
37,STU0462,AY2025,3,B.Tech CSE - AI & ML,SOCS,kandoli,19,Female,PCM,7.54,...,Average,High,Slot 1,5,Slot clash,8,?,20/10/2026 19:57,SOE20B028,Computation and Logic in Sanskrit Knowledge Sy...
41,STU0038,2023,7,B.Tech CSE - Core,NaN,Bidholi,21,MALE,NaN,5.8,...,-,High,Slot-1,4,Interests not matching available courses,8,3,2026-09-16 13:21:22,ECEG2059,Microprocessor and Microcontroller
43,STU0405,2025,3,B.Sc Biotechnology,SOB,Bidholi,19,Male,PCB,7.89 CGPA,...,Moderate,Medium,No Preference,2,No confusion,8,4,2026-10-17 21:44:52,LSPB1009,Foundations of Psychology-I
66,STU0640,2023,6,BBA- Marketing,SoB,Bidholi Campus,21,Male,Science (PCM),NaN,...,NaN,Hi,No Preference,2,No confusion,9,2,2026-08-02 01:02:59,MKTG3011,Rural marketing
78,STU0073,NaN,3,BBA - Human Resources,,Bidholi,21,Female,Comm,7.72,...,Easy,Medium,Slot 2,7,Unclear content,4.5,4,24/08/2026 22:03,SDIN2020,Foundations of Interaction & Human Beavior


Check duplicate rows in each dataset

In [21]:
#Display duplicate records
for name, df in datasets.items():
    duplicates = df[df.duplicated(keep=False)]
    print(f"\n---{name}---")
    print("Total duplicate rows:", len(duplicates))
    display(duplicates.head(5))  # Display first 5 duplicate records


---Students---
Total duplicate rows: 80


,Student_ID,Batch_Year,Semester,Branch,School,Campus,Age,Gender,Class12_Stream,CGPA,...,Preferred_Difficulty,Workload_Preference,Preferred_Slot,Confusion_Level,Confusion_Reason,Selection_Confidence,Previous_Course_Satisfaction,Survey_Timestamp,Selected_Course_Code,Selected_Exploratory_Course
14,STU0811,2023,6,BBA - FINANCE,SOB,kandoli,21,M,Arts/Humanities,"7,66",...,Easy,Medium,No Preference,3,Not Available,8/10,3,2026-09-19 01:52:48,SCS20A001,Data Visualization
16,STU0015,2024,5,B.Tech Applied Petroleum Engineering,SOAE,Bidholi,20,MALE,PCM,"7,31",...,Moderate,HIGH,No Preference,7,Too many options,4,3,23-09-26,LSHP3038,"History of Modern India, 1857-1947"
37,STU0462,AY2025,3,B.Tech CSE - AI & ML,SOCS,kandoli,19,Female,PCM,7.54,...,Average,High,Slot 1,5,Slot clash,8,?,20/10/2026 19:57,SOE20B028,Computation and Logic in Sanskrit Knowledge Sy...
41,STU0038,2023,7,B.Tech CSE - Core,NaN,Bidholi,21,MALE,NaN,5.8,...,-,High,Slot-1,4,Interests not matching available courses,8,3,2026-09-16 13:21:22,ECEG2059,Microprocessor and Microcontroller
43,STU0405,2025,3,B.Sc Biotechnology,SOB,Bidholi,19,Male,PCB,7.89 CGPA,...,Moderate,Medium,No Preference,2,No confusion,8,4,2026-10-17 21:44:52,LSPB1009,Foundations of Psychology-I



---Courses---
Total duplicate rows: 0


,Course_ID,AY,School,Sem,Course Code,Course,CREDIT,Minor Stream,Slot,Schedule,Remarks,Primary_Domain,Secondary_Domain,Difficulty_Level,Workload_Level,Teaching_Style,Technical_Prerequisite,Popularity_Index,Skill_Tags,Typical_Career_Paths



---Interactions---
Total duplicate rows: 228


,Interaction_ID,Student_ID,Course_ID,Course_Code,Interaction_Type,Interaction_Date,Time_Spent_Minutes,Recommended_By,Rating_Given,Suitability_Label
69,INT03480,STU0570,C026,CLCC1002,viewed,2026-07-24 12:22,2,Senior,NaN,Medium
86,INT00358,STU0062,C009,LSBA1006,viewed,2026-08-10 03:23,2,Senior student,NaN,Low
99,INT00098,STU0017,C121,CLCC4025,Shortlisted,2026-08-17 06:16:40,23,Faculty/Advisor,NaN,High
107,INT03675,STU0602,C094,PEAU2018,Viewed,30/09/2026,2,Senior,NaN,Low
122,INT03338,STU0546,C137,DIGB3008P,Shortlisted,00/00/0000,5,Friend,NaN,Medium



---Dictionary---
Total duplicate rows: 0


,Sheet,Column,Description,Expected_Data_Type,Valid_Range_Or_Allowed_Values,Nullable,Role,Notes,Injected_Issue_Types_In_Raw_Data


Generate dataset audit summary

In [25]:
audit_summary = []
for name, df in datasets.items():
    audit_summary.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns":df.shape[1],
        "Missing Values": df.isnull().sum().sum(),
        "Duplicate Rows": df.duplicated().sum(),
        "Total Cells": df.size
    })

audit_df = pd.DataFrame(audit_summary)
display(audit_df)

,Dataset,Rows,Columns,Missing Values,Duplicate Rows,Total Cells
0,Students,1020,33,1455,40,33660
1,Courses,178,20,288,0,3560
2,Interactions,5869,10,5104,114,58690
3,Dictionary,63,9,54,0,567


Save audit Summary

In [26]:
audit_df.to_csv("Dataset_Audit_Summary.csv", index= False)
print("Audit summary saved successfully!")

Audit summary saved successfully!
